# Quantitative Multi-Asset Screener

This notebook builds a quantitative screener for the Reto Actinver investment universe.

The methodology:

1. Loads historical data from PostgreSQL.
2. Uses adjusted prices to calculate returns and momentum.
3. Applies freshness, data-quality and liquidity filters.
4. Standardizes factors within each asset category.
5. Computes a baseline multi-factor score.
6. Ranks instruments within STOCK, ETF and FUND categories.
7. Selects the top percentile as the candidate pool.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sqlalchemy import text


PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


from src.database import get_engine
from src.features import calculate_features
from src.screener import (
    build_universe,
    select_candidates
)

In [2]:
engine = get_engine()

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Database connection successful.")

Database connection successful.


## 1. Load market data

The screener uses `adjusted_close` for return-based calculations in order to reduce distortions caused by stock splits and other corporate actions.

`close` is retained as the observed market closing price.

Run the updated notebook 01 first to create the maintenance table and recover missing
adjusted history. Assets with missing/invalid stored prices or unresolved rejected dates
are temporarily excluded. The pipeline's final reports explain what needs recovery.
We retain missing observations and use `pct_change(fill_method=None)` so a gap cannot
silently become a zero return.

In [3]:
prices = pd.read_sql(
    """
    SELECT
        p.date,
        p.asset_id,
        a.actinver_symbol,
        a.asset_type,
        p.close,
        p.adjusted_close,
        p.volume,
        COALESCE(r.pending_rejections, 0) AS pending_rejections
    FROM prices p
    JOIN assets a
        ON p.asset_id = a.asset_id
    LEFT JOIN (
        SELECT asset_id, COUNT(*) AS pending_rejections
        FROM public.price_rejections
        GROUP BY asset_id
    ) r ON r.asset_id = p.asset_id
    ORDER BY p.asset_id, p.date;
    """,
    engine
)

prices["date"] = pd.to_datetime(prices["date"])

In [4]:
prices = calculate_features(prices)

In [5]:
evaluation_date = (
    prices["date"].max()
)

universe = build_universe(
    prices=prices,
    evaluation_date=evaluation_date,
    momentum_weight=1/3,
    volatility_weight=1/3,
    drawdown_weight=1/3
)

candidate_pool = select_candidates(
    universe,
    cutoff=0.90
)

In [6]:
ranking = (
    universe
    .sort_values(
        [
            "asset_type",
            "category_rank"
        ]
    )
    .copy()
)

In [7]:
display(
    candidate_pool[
        [
            "actinver_symbol",
            "asset_type",
            "momentum_5d",
            "momentum_20d",
            "momentum_60d",
            "volatility_20d_ann",
            "current_drawdown",
            "score",
            "category_rank",
            "category_percentile"
        ]
    ]
)

,actinver_symbol,asset_type,momentum_5d,momentum_20d,momentum_60d,volatility_20d_ann,current_drawdown,score,category_rank,category_percentile
308610,VGT,ETF,0.046307,0.144192,0.127053,0.145041,0.000000,0.870302,1.0,1.000000
262736,SHV,ETF,0.028041,0.076993,0.045595,0.094412,-0.005520,0.704575,2.0,0.975000
245315,QQQ,ETF,0.031309,0.123293,0.074263,0.136006,0.000000,0.698983,3.0,0.950000
274501,SPYM,ETF,0.024945,0.071691,0.064529,0.109454,-0.002424,0.659718,4.0,0.925000
315403,VTI,ETF,0.027165,0.070502,0.055739,0.111788,0.000000,0.648560,5.0,0.900000
13164,ACTI500B,FUND,0.025293,0.065775,0.051184,0.088935,-0.001193,0.430369,1.0,1.000000
27828,ACTIRVTB,FUND,0.016442,0.037490,0.028208,0.042241,-0.000244,0.355120,2.0,0.944444
295253,TX,STOCK,0.045312,0.045205,0.335495,0.208347,0.000000,0.940309,1.0,1.000000
291855,TSMN,STOCK,0.078411,0.228213,0.123104,0.276782,0.000000,0.925430,2.0,0.992481
216452,NVDA,STOCK,0.067201,0.098350,0.191058,0.265913,0.000000,0.832815,3.0,0.984962


In [8]:
candidate_pool[
    "asset_type"
].value_counts()

asset_type
STOCK    14
ETF       5
FUND      2
Name: count, dtype: int64

In [9]:
print(
    f"Evaluation date: "
    f"{evaluation_date.date()}"
)

print(
    f"Eligible assets: "
    f"{len(universe)}"
)

print(
    f"Candidate assets: "
    f"{len(candidate_pool)}"
)

Evaluation date: 2026-10-02
Eligible assets: 191
Candidate assets: 21


In [10]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

candidate_pool.to_csv(
    OUTPUT_DIR / "candidate_pool.csv",
    index=False
)

ranking.to_csv(
    OUTPUT_DIR / "screener_ranking.csv",
    index=False
)

print(
    f"Candidate pool saved to: "
    f"{OUTPUT_DIR / 'candidate_pool.csv'}"
)

Candidate pool saved to: c:\Users\alexi\OneDrive\Desktop\ProyectosPRO\bmv-quant-portfolio-system\data\processed\candidate_pool.csv
